<a href="https://colab.research.google.com/github/ysuter/FHNW-BAI-ComputerVision/blob/main/W15_DomainShift.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Woche 15 – Demo: Domain Shift
## Computer Vision · BAI · FS 2026

**Worum es heute geht:** Ein vortrainiertes Bildklassifikations-Modell trifft im Labor sauber – wir schauen, was passiert, wenn das Bild *nicht mehr aussieht wie der Trainingsdatensatz*.

**Drei Experimente:**
1. **Common Corruptions** – Rauschen, Unschärfe, Helligkeitsänderungen (ImageNet-C-Stil)
2. **Wetter / Witterung** – Schnee, Regen, Nebel als Stellvertreter für Outdoor-Deployment
3. **Domain Gap** – Foto → Skizze / Cartoon

Wir verwenden ein ResNet-50, vortrainiert auf ImageNet, und beobachten Confidence-Verfall und Klassen-Wechsel.

---
## 1. Setup

In [ ]:
# Bibliotheken (in Colab vorinstalliert, ausser scikit-image für manche Korruptionen)
import torch
import torchvision.models as models
import torchvision.transforms as T
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter, ImageEnhance
import requests
from io import BytesIO

plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')


In [ ]:
# ImageNet-Klassenlabels laden (1000 Klassen)
LABELS_URL = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
imagenet_labels = requests.get(LABELS_URL).text.strip().split('\n')
print(f'Anzahl Klassen: {len(imagenet_labels)}')
print(f'Beispiel: Klasse 207 = "{imagenet_labels[207]}"')


In [ ]:
# Vortrainiertes Modell laden
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model.eval()
model = model.to(device)
print('ResNet-50 geladen.')

# Standard-Preprocessing für ImageNet-Modelle
preprocess = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225]),
])


In [ ]:
def predict(img_pil, topk=3):
    """Gibt Top-k (Klasse, Confidence) für ein PIL-Bild zurück."""
    x = preprocess(img_pil).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(x)
        probs = torch.softmax(logits, dim=1)[0]
    top_probs, top_idx = probs.topk(topk)
    return [(imagenet_labels[idx.item()], prob.item())
            for idx, prob in zip(top_idx, top_probs)]

def show_image_with_pred(img, title, ax):
    ax.imshow(img)
    ax.axis('off')
    preds = predict(img, topk=3)
    top_class, top_conf = preds[0]
    short_title = f'{title}\n→ {top_class} ({top_conf:.0%})'
    ax.set_title(short_title, fontsize=9)
    return preds


---
## 2. Referenzbild – Goldstandard im Labor

Wir laden das offizielle PyTorch-Hub-Hundebild (verfügbar im pytorch/hub-Repo auf GitHub). Ein Hund eignet sich gut, weil ImageNet sehr viele Hunde-Klassen hat – wir sehen, ob das Modell zumindest in der Hunde-Familie bleibt, wenn es schon falsch liegt.

> **Hinweis:** Bilder von Wikimedia (`upload.wikimedia.org`) werden in Colab oft blockiert, wenn man sie ohne User-Agent-Header lädt. GitHub-Raw-URLs und Unsplash-Direktlinks funktionieren dagegen zuverlässig. Im Student-Notebook gibt es einen Helfer, der den User-Agent setzt – hier verwenden wir die einfachste robuste Quelle.


In [ ]:
# Referenzbild laden – Hundebild aus dem offiziellen PyTorch-Hub
IMG_URL = 'https://raw.githubusercontent.com/pytorch/hub/master/images/dog.jpg'
response = requests.get(IMG_URL)
response.raise_for_status()
img_clean = Image.open(BytesIO(response.content)).convert('RGB')

# Auf saubere Grösse bringen
img_clean = img_clean.resize((512, 384))

# Vorhersage zeigen
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
preds = show_image_with_pred(img_clean, 'Referenz', ax)
plt.tight_layout()
plt.show()

print('\nTop-3 Vorhersagen:')
for i, (cls, conf) in enumerate(preds, 1):
    print(f'  {i}. {cls:40s} {conf:.2%}')


---
## 3. Experiment 1 – Common Corruptions (ImageNet-C-Stil)

ImageNet-C definiert 15 Korruptions-Typen × 5 Schweregrade. Wir implementieren hier vereinfachte Versionen von vier Klassikern:

- **Gaussian Noise** – Sensorrauschen bei wenig Licht
- **Gaussian Blur** – unscharfes Bild (defokussiert, verstaubte Linse)
- **Brightness Reduction** – Bild bei Dämmerung
- **JPEG Compression** – starke Komprimierung wie bei Bandbreitenproblemen

Für jede Korruption probieren wir 3 Schweregrade. Beobachte: **wo bricht die Confidence ein und wechselt die Klasse?**


In [ ]:
def add_gaussian_noise(img, sigma):
    arr = np.array(img).astype(np.float32)
    noise = np.random.normal(0, sigma * 255, arr.shape)
    arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)

def apply_blur(img, radius):
    return img.filter(ImageFilter.GaussianBlur(radius=radius))

def reduce_brightness(img, factor):
    return ImageEnhance.Brightness(img).enhance(factor)

def jpeg_compress(img, quality):
    buf = BytesIO()
    img.save(buf, 'JPEG', quality=quality)
    buf.seek(0)
    return Image.open(buf).convert('RGB')

CORRUPTIONS = {
    'Gauss-Rauschen': [(add_gaussian_noise, s, f'σ={s}') for s in [0.05, 0.15, 0.30]],
    'Unschärfe':      [(apply_blur, r, f'r={r}px') for r in [2, 5, 10]],
    'Dunkelheit':     [(reduce_brightness, f, f'×{f}') for f in [0.5, 0.2, 0.05]],
    'JPEG-Komp.':     [(jpeg_compress, q, f'q={q}') for q in [30, 10, 5]],
}


In [ ]:
# Grid aller Korruptionen rendern
fig, axes = plt.subplots(4, 4, figsize=(16, 14))

# Erste Spalte: Referenz
for row in range(4):
    show_image_with_pred(img_clean, 'Referenz', axes[row, 0])

# Restliche Spalten: 3 Schweregrade pro Zeile
results = {}
for row, (name, severities) in enumerate(CORRUPTIONS.items()):
    results[name] = []
    for col, (fn, level, label) in enumerate(severities, start=1):
        img_corr = fn(img_clean, level)
        preds = show_image_with_pred(img_corr, f'{name}\n{label}', axes[row, col])
        results[name].append(preds)

plt.tight_layout()
plt.show()


In [ ]:
# Confidence-Verfall als Tabelle ausgeben
print(f'{"Korruption":<18}{"Level 1":<15}{"Level 2":<15}{"Level 3":<15}')
print('-' * 65)

# Referenz-Confidence
ref_preds = predict(img_clean, topk=1)
ref_class = ref_preds[0][0]
ref_conf = ref_preds[0][1]
print(f'{"Referenz":<18}{ref_conf:>6.1%}  ({ref_class})')
print()

for name, preds_list in results.items():
    row = f'{name:<18}'
    for preds in preds_list:
        top_class, top_conf = preds[0]
        match = '✓' if top_class == ref_class else '✗'
        row += f'{top_conf:>5.0%} {match}      '
    print(row)


---
## 4. Experiment 2 – Witterung

Outdoor-CV-Systeme (Verkehrsüberwachung, Wildkameras, Drohnen) sehen Wetter, das im Trainingsdatensatz unterrepräsentiert war. Wir simulieren:

- **Schnee** – additives helles Rauschen + Helligkeit
- **Nebel** – additiver weisser Schleier
- **Regen** – diagonale helle Streifen


In [ ]:
def add_snow(img, intensity=0.3):
    arr = np.array(img).astype(np.float32) / 255.0
    snow_mask = np.random.random(arr.shape[:2])
    snow_mask = (snow_mask > (1 - intensity)).astype(np.float32)
    snow_mask = np.stack([snow_mask] * 3, axis=2)
    arr = arr + snow_mask * 0.8
    return Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))

def add_fog(img, intensity=0.4):
    arr = np.array(img).astype(np.float32) / 255.0
    fog = np.ones_like(arr) * 0.85
    arr = arr * (1 - intensity) + fog * intensity
    return Image.fromarray((np.clip(arr, 0, 1) * 255).astype(np.uint8))

def add_rain(img, n_streaks=200, length=15):
    arr = np.array(img).copy()
    h, w = arr.shape[:2]
    for _ in range(n_streaks):
        x = np.random.randint(0, w - length)
        y = np.random.randint(0, h - length)
        for i in range(length):
            if y + i < h and x + i // 3 < w:
                arr[y + i, x + i // 3] = [220, 220, 230]
    return Image.fromarray(arr)

WEATHER = {
    'Schnee leicht': (add_snow, 0.10),
    'Schnee stark':  (add_snow, 0.30),
    'Nebel leicht':  (add_fog, 0.25),
    'Nebel stark':   (add_fog, 0.55),
    'Regen':         (add_rain, None),
}


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
show_image_with_pred(img_clean, 'Referenz', axes[0, 0])

weather_items = list(WEATHER.items())
for idx, (name, (fn, param)) in enumerate(weather_items):
    row, col = (idx + 1) // 3, (idx + 1) % 3
    img_w = fn(img_clean) if param is None else fn(img_clean, param)
    show_image_with_pred(img_w, name, axes[row, col])

plt.tight_layout()
plt.show()


---
## 5. Experiment 3 – Domain Gap (Foto → Skizze / Cartoon)

ImageNet-Sketch und ImageNet-R zeigen: Modelle, die auf Fotos trainiert sind, verlieren massiv, sobald die *Bilddomäne* wechselt – auch wenn die Klasse offensichtlich gleich bleibt. Wir simulieren das mit klassischer Bildverarbeitung.


In [ ]:
def to_sketch(img):
    """Pencil-Sketch-Effekt: Graustufen + invert + blur + dodge."""
    arr = np.array(img.convert('L')).astype(np.float32)
    inverted = 255 - arr
    blurred = np.array(Image.fromarray(inverted.astype(np.uint8))
                         .filter(ImageFilter.GaussianBlur(15))).astype(np.float32)
    sketch = arr * 255.0 / (256.0 - blurred + 1e-3)
    sketch = np.clip(sketch, 0, 255).astype(np.uint8)
    return Image.fromarray(sketch).convert('RGB')

def to_cartoon(img):
    """Cartoon-Effekt: Posterisierung + verstärkte Kanten."""
    from PIL import ImageOps
    posterized = ImageOps.posterize(img, 3)
    enhancer = ImageEnhance.Color(posterized)
    return enhancer.enhance(1.8)

def to_grayscale_only(img):
    return img.convert('L').convert('RGB')

def to_negative(img):
    arr = 255 - np.array(img)
    return Image.fromarray(arr)

DOMAINS = {
    'Skizze (Bleistift)': to_sketch,
    'Cartoon':            to_cartoon,
    'Schwarz-Weiss':      to_grayscale_only,
    'Farb-Negativ':       to_negative,
}


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(20, 5))
show_image_with_pred(img_clean, 'Referenz', axes[0])

for idx, (name, fn) in enumerate(DOMAINS.items(), start=1):
    img_d = fn(img_clean)
    show_image_with_pred(img_d, name, axes[idx])

plt.tight_layout()
plt.show()


**Was beobachten wir hier typischerweise?**

- **Schwarz-Weiss** wird oft noch erkannt (Foto-Domäne ist nah)
- **Skizze und Cartoon** kippen meist – die Textur, die das Modell als Feature nutzt, fehlt
- **Farb-Negativ** kann erstaunlich katastrophal sein (Textur stimmt, aber alle Farben sind invertiert – das Modell hat das nie gesehen)

Das ist der Kern des **Texture-Bias** in CNNs (Geirhos et al., 2019): ImageNet-Modelle entscheiden viel stärker über Textur als über Form. Eine Skizze hat dieselbe Form, aber komplett andere Textur.


---
## 6. Bonus – Confidence-Kurve bei graduellem Domain Shift

Statt diskreter Stufen: was passiert, wenn wir den Schweregrad *kontinuierlich* erhöhen? Hier exemplarisch für Gauss-Rauschen.


In [ ]:
sigmas = np.linspace(0, 0.5, 11)
confidences = []
classes = []

for sigma in sigmas:
    img_n = add_gaussian_noise(img_clean, sigma) if sigma > 0 else img_clean
    preds = predict(img_n, topk=1)
    classes.append(preds[0][0])
    confidences.append(preds[0][1])

fig, ax = plt.subplots(figsize=(10, 5))
ref_class = classes[0]
colors = ['green' if c == ref_class else 'red' for c in classes]
ax.bar(range(len(sigmas)), confidences, color=colors, alpha=0.7)
ax.set_xticks(range(len(sigmas)))
ax.set_xticklabels([f'{s:.2f}' for s in sigmas])
ax.set_xlabel('Gauss-Rauschen σ')
ax.set_ylabel('Confidence (Top-1)')
ax.set_title(f'Confidence-Verfall – grün = Klasse stabil ({ref_class}), rot = Klasse gewechselt')
ax.set_ylim(0, 1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print('\nKlassen-Verlauf:')
for s, c, conf in zip(sigmas, classes, confidences):
    marker = '✓' if c == ref_class else '✗'
    print(f'  σ={s:.2f}: {marker} {c:30s} ({conf:.0%})')


---
## 7. Diskussion in der Vorlesung

1. **Wo im Lebenszyklus** lässt sich Distribution Shift bekämpfen? Daten, Training, Deployment oder Monitoring?
2. **Confidence reicht nicht.** Selbst bei massiver Korruption gibt das Modell oft hohe Confidence-Werte für die *falsche* Klasse. Warum ist das gefährlich, und welche Strategien helfen (Stichworte: Out-of-Distribution-Detection, Calibration)?
3. **Texture vs. Shape Bias.** ImageNet-Modelle nutzen oft Textur statt Form. Welche Auswirkungen hat das auf Anwendungen, die in ungewohnten Bilddomänen laufen (Industrie-Inspektion mit anderen Materialien, medizinische Bildgebung)?
4. **Test-Augmentation.** Wenn wir vor dem Deployment mit den Korruptionen aus dieser Demo testen würden – was würden wir messen, was würden wir übersehen?

---

*Hinweis: Diese Demo verwendet einfache Korruptionen. Der offizielle ImageNet-C-Benchmark nutzt 15 Typen mit standardisierter Implementierung